# Entrenamiento de EfficientNet-B0 con BUS-BRA

Notebook autocontenido para Google Colab. Entrena un clasificador binario benigno/maligno a partir de un manifiesto agrupado por paciente.

**Advertencia metodológica:** el protocolo V2 reserva BUS-BRA para validación externa del modelo entrenado en BUSI. Entrenar con BUS-BRA constituye un experimento diferente y debe reportarse como tal.

## 1. Activar GPU
En Colab: **Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU**.

In [ ]:
import sys, subprocess, importlib

required = {
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'yaml': 'pyyaml',
    'PIL': 'pillow',
}
missing = []
for module, package in required.items():
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(package)
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import torch, torchvision
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Activá una GPU antes de entrenar.')

## 2. Montar Google Drive
Guardá las imágenes, el manifiesto y los resultados en Drive para que sobrevivan al reinicio de Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuración
El notebook usa directamente los CSV oficiales de BUS-BRA. `5-fold-cv.csv` contiene `ID`, `Pathology` y `kFold`; `bus_data.csv` aporta metadatos opcionales. Se reserva un fold para test, otro para validación y los restantes para entrenamiento.

In [ ]:
from pathlib import Path

# CAMBIAR ESTAS RUTAS SEGÚN TU DRIVE
IMAGE_ROOT = Path('/content/drive/MyDrive/BUS-BRA/Images')
FOLD_CSV_PATH = Path('/content/drive/MyDrive/BUS-BRA/5-fold-cv.csv')
METADATA_CSV_PATH = Path('/content/drive/MyDrive/BUS-BRA/bus_data.csv')
OUTPUT_DIR = Path('/content/drive/MyDrive/BUS-BRA/resultados_efficientnet_b0')
TEST_FOLD = 1
VALIDATION_FOLD = 2
IMAGE_EXTENSION = '.png'

SEED = 2026
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2
EPOCHS = 20
FREEZE_BACKBONE_EPOCHS = 2
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 4
POSITIVE_LABELS = {'malignant', '1'}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Imágenes:', IMAGE_ROOT)
print('CSV de folds:', FOLD_CSV_PATH)
print('Metadatos:', METADATA_CSV_PATH)
print('Resultados:', OUTPUT_DIR)

## 4. Construir y validar las particiones oficiales
La celda convierte el CSV oficial a un manifiesto interno. Se detiene si faltan columnas, folds o imágenes, o si un paciente aparece en folds diferentes.

In [ ]:
import pandas as pd

if not FOLD_CSV_PATH.is_file():
    raise FileNotFoundError(f'No se encontró el CSV de folds: {FOLD_CSV_PATH}')
if not IMAGE_ROOT.is_dir():
    raise FileNotFoundError(f'No se encontró la carpeta de imágenes: {IMAGE_ROOT}')

folds = pd.read_csv(FOLD_CSV_PATH)
required_columns = {'ID', 'Pathology', 'kFold'}
missing_columns = required_columns - set(folds.columns)
if missing_columns:
    raise ValueError(f'Faltan columnas en el CSV oficial: {sorted(missing_columns)}')
if TEST_FOLD == VALIDATION_FOLD:
    raise ValueError('TEST_FOLD y VALIDATION_FOLD deben ser diferentes.')
folds['kFold'] = pd.to_numeric(folds['kFold'], errors='raise').astype(int)
available_folds = set(folds['kFold'].unique())
if TEST_FOLD not in available_folds or VALIDATION_FOLD not in available_folds:
    raise ValueError(f'Folds disponibles: {sorted(available_folds)}')

manifest = folds.copy()
manifest['image_path'] = manifest['ID'].astype(str) + IMAGE_EXTENSION
manifest['label'] = manifest['Pathology'].astype(str).str.lower().str.strip()
manifest['patient_id'] = manifest['ID'].astype(str).str.rsplit('-', n=1).str[0]
manifest['split'] = 'train'
manifest.loc[manifest['kFold'] == VALIDATION_FOLD, 'split'] = 'val'
manifest.loc[manifest['kFold'] == TEST_FOLD, 'split'] = 'test'

patient_fold_counts = manifest.groupby('patient_id')['kFold'].nunique()
leaked_patients = patient_fold_counts[patient_fold_counts > 1]
if not leaked_patients.empty:
    raise ValueError(f'Pacientes asignados a folds diferentes: {leaked_patients.index.tolist()[:10]}')

manifest['target'] = manifest['label'].isin(POSITIVE_LABELS).astype(int)
manifest['resolved_path'] = manifest['image_path'].map(lambda name: IMAGE_ROOT / name)
missing_images = manifest.loc[~manifest['resolved_path'].map(Path.is_file), 'resolved_path']
if len(missing_images):
    raise FileNotFoundError(f'Faltan {len(missing_images)} imágenes. Primeras rutas: {missing_images.head().tolist()}')
manifest.drop(columns='resolved_path').to_csv(OUTPUT_DIR / 'resolved_manifest.csv', index=False)

if METADATA_CSV_PATH.is_file():
    bus_data = pd.read_csv(METADATA_CSV_PATH)
    print('Columnas disponibles en bus_data.csv:', bus_data.columns.tolist())
else:
    print('Aviso: bus_data.csv no se encontró. No es obligatorio para entrenar con Pathology.')
summary = manifest.groupby(['split', 'target']).size().unstack(fill_value=0)
display(summary.rename(columns={0: 'benigno', 1: 'maligno'}))
print('Imágenes:', len(manifest))
print('Pacientes:', manifest['patient_id'].nunique())

## 5. Dataset y transformaciones
Las imágenes se convierten a escala de grises, se repiten en tres canales, se rellenan para conservar la proporción y se normalizan con los parámetros de ImageNet.

In [ ]:
import random
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import functional as TF

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class SquarePad:
    def __call__(self, image):
        width, height = image.size
        side = max(width, height)
        left = (side - width) // 2
        top = (side - height) // 2
        return TF.pad(image, [left, top, side-width-left, side-height-top], fill=0)

def make_transform(train=False):
    ops = [transforms.Grayscale(num_output_channels=3), SquarePad(), transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True)]
    if train:
        ops += [transforms.RandomHorizontalFlip(0.5), transforms.RandomRotation(8, fill=0)]
    ops += [transforms.ToTensor(), transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))]
    return transforms.Compose(ops)

class BusBraDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        path = Path(str(row.image_path))
        if not path.is_absolute():
            path = IMAGE_ROOT / path
        if not path.is_file():
            raise FileNotFoundError(path)
        with Image.open(path) as source:
            image = source.convert('L')
        return self.transform(image), torch.tensor(float(row.target), dtype=torch.float32), str(path), str(row.patient_id)

seed_everything(SEED)
frames = {split: manifest[manifest.split == split].copy() for split in ('train', 'val', 'test')}
datasets = {split: BusBraDataset(frame, make_transform(train=split == 'train')) for split, frame in frames.items()}
loaders = {
    split: DataLoader(ds, batch_size=BATCH_SIZE, shuffle=split == 'train', num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
    for split, ds in datasets.items()
}
print({split: len(ds) for split, ds in datasets.items()})

## 6. EfficientNet-B0 y ciclo de entrenamiento

In [ ]:
import json
from sklearn.metrics import roc_auc_score, average_precision_score
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

device = torch.device('cuda')
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)
in_features = model.classifier[1].in_features
model.classifier = torch.nn.Sequential(torch.nn.Dropout(0.2), torch.nn.Linear(in_features, 1))
model.to(device)

for parameter in model.features.parameters():
    parameter.requires_grad = False

train_targets = frames['train']['target']
positives = int(train_targets.sum())
negatives = int((1-train_targets).sum())
if positives == 0 or negatives == 0:
    raise ValueError('Train debe contener ambas clases.')
pos_weight = torch.tensor([negatives / positives], device=device)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler(enabled=True)

def run_epoch(loader, training=False):
    model.train(training)
    losses, labels, scores = [], [], []
    for images, target, _, _ in loader:
        images, target = images.to(device, non_blocking=True), target.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.cuda.amp.autocast(enabled=True):
                logits = model(images).squeeze(1)
                loss = criterion(logits, target)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        losses.append(loss.item())
        labels.extend(target.detach().cpu().numpy())
        scores.extend(torch.sigmoid(logits).detach().cpu().numpy())
    labels, scores = np.asarray(labels), np.asarray(scores)
    return {'loss': float(np.mean(losses)), 'auc': float(roc_auc_score(labels, scores)), 'ap': float(average_precision_score(labels, scores)), 'labels': labels, 'scores': scores}

best_auc = -np.inf
wait = 0
history = []
checkpoint_path = OUTPUT_DIR / 'best_model.pt'

for epoch in range(1, EPOCHS + 1):
    if epoch == FREEZE_BACKBONE_EPOCHS + 1:
        for parameter in model.features.parameters():
            parameter.requires_grad = True
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE/10, weight_decay=WEIGHT_DECAY)
    train_metrics = run_epoch(loaders['train'], training=True)
    with torch.inference_mode():
        val_metrics = run_epoch(loaders['val'])
    row = {'epoch': epoch, 'train_loss': train_metrics['loss'], 'val_loss': val_metrics['loss'], 'val_auc': val_metrics['auc'], 'val_ap': val_metrics['ap']}
    history.append(row)
    print(json.dumps(row, ensure_ascii=False))
    if val_metrics['auc'] > best_auc:
        best_auc = val_metrics['auc']
        wait = 0
        torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch, 'val_auc': best_auc, 'seed': SEED}, checkpoint_path)
    else:
        wait += 1
        if wait >= EARLY_STOPPING_PATIENCE:
            print('Detención temprana.')
            break

history = pd.DataFrame(history)
history.to_csv(OUTPUT_DIR / 'history.csv', index=False)
print('Mejor checkpoint:', checkpoint_path)

## 7. Curvas de entrenamiento

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.epoch, history.train_loss, label='train')
axes[0].plot(history.epoch, history.val_loss, label='validación')
axes[0].set(xlabel='Época', ylabel='Loss', title='Pérdida')
axes[0].legend()
axes[1].plot(history.epoch, history.val_auc, label='AUC-ROC')
axes[1].plot(history.epoch, history.val_ap, label='Average precision')
axes[1].set(xlabel='Época', ylabel='Métrica', title='Validación', ylim=(0, 1))
axes[1].legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=160, bbox_inches='tight')
plt.show()

## 8. Evaluación final en test
Esta celda abre el test sólo después de seleccionar el checkpoint por validación. Usa el umbral 0,5 como referencia; los umbrales clínicamente orientados deben seleccionarse exclusivamente en validación.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, RocCurveDisplay, PrecisionRecallDisplay

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_labels, all_scores, all_paths, all_patients = [], [], [], []
with torch.inference_mode():
    for images, target, paths, patients in loaders['test']:
        logits = model(images.to(device)).squeeze(1)
        all_labels.extend(target.numpy())
        all_scores.extend(torch.sigmoid(logits).cpu().numpy())
        all_paths.extend(paths)
        all_patients.extend(patients)

all_labels = np.asarray(all_labels, dtype=int)
all_scores = np.asarray(all_scores)
predictions = (all_scores >= 0.5).astype(int)
print('AUC-ROC:', roc_auc_score(all_labels, all_scores))
print('Average precision:', average_precision_score(all_labels, all_scores))
print('Matriz de confusión:\n', confusion_matrix(all_labels, predictions))
print(classification_report(all_labels, predictions, target_names=['benigno', 'maligno'], digits=3))

pd.DataFrame({'image_path': all_paths, 'patient_id': all_patients, 'target': all_labels, 'score_malignancy': all_scores, 'prediction_0_5': predictions}).to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
RocCurveDisplay.from_predictions(all_labels, all_scores, ax=axes[0])
PrecisionRecallDisplay.from_predictions(all_labels, all_scores, ax=axes[1])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'test_curves.png', dpi=160, bbox_inches='tight')
plt.show()

## 9. Descargar resultados principales

In [ ]:
from google.colab import files

print('Todos los resultados quedaron guardados en:', OUTPUT_DIR)
# Descomentá si querés descargar el checkpoint al equipo local.
# files.download(str(checkpoint_path))